# Train `internvl` — PAN924 dental report VLM (Colab · A100)

**Model:** `OpenGVLab/InternVL3-8B`  ·  **Framework:** ms-swift (LoRA, bf16)

All 5 notebooks share the same settings, so the models are comparable. Checkpoints are saved on **Google Drive**, so if Colab disconnects you just **re-run the train cell and it continues** from the last checkpoint.

## 1. Check the GPU
`Runtime -> Change runtime type -> A100 GPU`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install ms-swift and the per-model dependencies

In [ ]:
# Pinned to the 3.x line these notebooks were built against, so the CLI flags below stay valid.
# After your first successful run, replace this with the exact version printed at the bottom
# of this cell (e.g. ms-swift==3.x.y) to lock the run down completely.
%pip install -q "ms-swift>=3.2,<4.0" accelerate
%pip install -q -U qwen_vl_utils timm einops sentencepiece hf_transfer
# flash-attn is OPTIONAL. It speeds up the A100 ~10-20%, but compiling it on Colab takes
# 15-30 min and often fails. ms-swift falls back to PyTorch SDPA (fast + reliable), which is
# what these notebooks use by default (ATTN_IMPL stays None). Uncomment ONLY for extra speed:
# %pip install -q flash-attn --no-build-isolation
import os
# ms-swift defaults to ModelScope (slow from outside China). Force Hugging Face + fast transfer.
# These MUST be set before importing swift / huggingface_hub.
os.environ["USE_HF"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
import swift, torch
assert torch.cuda.is_available(), 'No GPU. Set Runtime -> Change runtime type -> A100 GPU.'
print('ms-swift', swift.__version__, '| torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0), '| bf16 supported:', torch.cuda.is_bf16_supported())


## 3. Mount Google Drive (this is what makes training resumable)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Get the dataset onto the VM's local disk
Reading 4620 small images straight off Drive every epoch is slow and stalls, so we put the data on the **local** VM disk `/content/pan924` and train from there. Only checkpoints go back to Drive (step 6).

This cell uses whatever you have on Drive, **preferring the zip** because it's much faster:
- **`Thesis/pan924_vlm.zip`** → unzipped locally (~1–3 min, one big file = fast). **Recommended.**
- else **`Thesis/pan924_vlm/vlm_report_dataset/`** (extracted folder) → copied file-by-file (slower).

After a disconnect just re-run this cell.

In [ ]:
import os, shutil, zipfile
DRIVE_ZIP    = '/content/drive/MyDrive/Thesis/pan924_vlm.zip'   # fast path: one big file
DRIVE_FOLDER = '/content/drive/MyDrive/Thesis/pan924_vlm'       # fallback: already-extracted folder
REPO_DIR     = '/content/pan924'                                # local copy used for training

# Diagnostics first, so a wrong path / unmounted Drive is obvious.
print('Drive mounted?  ', os.path.isdir('/content/drive/MyDrive'))
print('zip on Drive?   ', os.path.exists(DRIVE_ZIP))
print('folder on Drive?', os.path.isdir(os.path.join(DRIVE_FOLDER, 'vlm_report_dataset')))
if os.path.isdir('/content/drive/MyDrive/Thesis'):
    print('Thesis/ contains:', os.listdir('/content/drive/MyDrive/Thesis'))

# Check the actual target FILE (not just the folder), so a partial leftover dir is re-filled.
train_jsonl = os.path.join(REPO_DIR, 'vlm_report_dataset', 'converted', 'qwen', 'train.jsonl')
os.makedirs(REPO_DIR, exist_ok=True)
if not os.path.exists(train_jsonl):
    if os.path.exists(DRIVE_ZIP):
        print('Unzipping from Drive (zipfile, no shell)...')
        with zipfile.ZipFile(DRIVE_ZIP) as z:
            z.extractall(REPO_DIR)
    elif os.path.isdir(os.path.join(DRIVE_FOLDER, 'vlm_report_dataset')):
        print('No zip - copying the extracted folder from Drive (slower)...')
        shutil.copytree(os.path.join(DRIVE_FOLDER, 'vlm_report_dataset'),
                        os.path.join(REPO_DIR, 'vlm_report_dataset'), dirs_exist_ok=True)
    else:
        raise FileNotFoundError('Data not on Drive. Checked:\n  ' + DRIVE_ZIP +
                                '\n  ' + os.path.join(DRIVE_FOLDER, 'vlm_report_dataset') +
                                '\nMount Drive (step 3) and check the folder/zip name.')
    print('Data ready on local disk.')

assert os.path.exists(train_jsonl), 'Still missing after extract: ' + train_jsonl
os.chdir(REPO_DIR)   # the image paths in the data are relative to here

import json
sample_image = json.loads(open(train_jsonl, encoding='utf-8').readline())['images'][0]
assert os.path.exists(sample_image), 'sample image not found: ' + sample_image
print('OK - data and images found. Working directory:', os.getcwd())


## 5. Settings
Effective batch = `PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS` = 1 x 16 = 16 (same as the 3090 run). If you have memory to spare, set batch=2 / accum=8 (still 16) to train ~2x faster.

In [ ]:
# ===== This model (the only part that differs between the 5 notebooks) =====
MODEL_NAME = "OpenGVLab/InternVL3-8B"
MODEL_KEY = "internvl"
USE_MAX_PIXELS = True
ATTN_IMPL = None      # None = ms-swift uses PyTorch SDPA (no flash-attn build needed); Phi uses 'eager'

# ===== Shared A100 settings (identical in all 5 notebooks = fair comparison) =====
TRAIN_TYPE = "lora"          # bf16 LoRA (A100 has the VRAM, no 4-bit needed)
TORCH_DTYPE = "bfloat16"
LORA_RANK = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = "all-linear"
FREEZE_VIT = "true"
NUM_EPOCHS = 2
LEARNING_RATE = "1e-4"
WEIGHT_DECAY = "0.1"
WARMUP_RATIO = "0.05"
LR_SCHEDULER = "cosine"
PER_DEVICE_BATCH_SIZE = 1    # safe on a 40GB A100. effective batch = 1 * 16 = 16 (same as the 3090 run)
GRAD_ACCUM_STEPS = 16        # if you DON'T hit OOM, set BATCH=2 / ACCUM=8 (still 16) to train ~2x faster
MAX_LENGTH = 4096
GRAD_CHECKPOINTING = "true"
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 12
SEED = 924
MAX_PIXELS = 1003520         # 1280*28*28 (higher than the 3090; lower first if OOM)

import os
# One folder PER MODEL on Drive, so the 5 models never overwrite each other's checkpoints/results.
OUTPUT_DIR = "/content/drive/MyDrive/Thesis/pan924_runs/" + MODEL_KEY
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Checkpoints + results for this model go to:", OUTPUT_DIR)


## 6. Download the base model (watch the progress bar)
Downloads the ~16GB base model to the VM with a **live progress bar**, so you can confirm it's really downloading (not hung). `hf_transfer` (step 2) makes it quick (~2-3 min). The train cell then loads it with no extra download. On a fresh VM after a disconnect, re-run this cell — it re-downloads (with progress) in a couple of minutes.

In [ ]:
# Download the base model HERE (in the notebook) so you see a LIVE progress bar and can tell
# it is really downloading, not frozen. It goes to the VM's local HF cache; the train cell then
# loads it with NO second silent download. On a fresh VM after a disconnect, just re-run this cell.
from huggingface_hub import snapshot_download
print("Downloading", MODEL_NAME, "- watch the progress bars below (hf_transfer = fast):")
local_path = snapshot_download(MODEL_NAME)
print("\nBase model ready at:", local_path)


## 7. Train (auto-resume)
Re-run this cell after a disconnect and it continues from the last checkpoint on Drive. The base model is already on the VM (step 6), so no download happens here.

In [ ]:
import os
import glob
import sys
import subprocess

def find_last_checkpoint(folder):
    """Return the newest checkpoint-N folder, or None if there is none yet."""
    last_path = None
    last_step = -1
    for path in glob.glob(os.path.join(folder, "checkpoint-*")):
        number_text = os.path.basename(path).replace("checkpoint-", "")
        if number_text.isdigit() and int(number_text) > last_step:
            last_step = int(number_text)
            last_path = path
    return last_path

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"          # stream logs live instead of buffering them
if USE_MAX_PIXELS:
    env["MAX_PIXELS"] = str(MAX_PIXELS)

command = [
    "swift", "sft",
    "--model", MODEL_NAME,
    "--dataset", "vlm_report_dataset/converted/qwen/train.jsonl",
    "--val_dataset", "vlm_report_dataset/converted/qwen/val.jsonl",
    "--split_dataset_ratio", "0",
    "--train_type", TRAIN_TYPE,
    "--torch_dtype", TORCH_DTYPE,
    "--lora_rank", str(LORA_RANK),
    "--lora_alpha", str(LORA_ALPHA),
    "--lora_dropout", str(LORA_DROPOUT),
    "--target_modules", TARGET_MODULES,
    "--freeze_vit", FREEZE_VIT,
    "--num_train_epochs", str(NUM_EPOCHS),
    "--learning_rate", LEARNING_RATE,
    "--weight_decay", WEIGHT_DECAY,
    "--warmup_ratio", WARMUP_RATIO,
    "--lr_scheduler_type", LR_SCHEDULER,
    "--per_device_train_batch_size", str(PER_DEVICE_BATCH_SIZE),
    "--per_device_eval_batch_size", "1",
    "--gradient_accumulation_steps", str(GRAD_ACCUM_STEPS),
    "--dataloader_num_workers", "4",
    "--max_length", str(MAX_LENGTH),
    "--gradient_checkpointing", GRAD_CHECKPOINTING,
    "--eval_strategy", "steps",
    "--eval_steps", str(EVAL_STEPS),
    "--save_strategy", "steps",
    "--save_steps", str(SAVE_STEPS),
    "--save_total_limit", str(SAVE_TOTAL_LIMIT),
    "--logging_steps", "5",
    "--seed", str(SEED),
    "--add_version", "false",
    "--output_dir", OUTPUT_DIR,
]

if ATTN_IMPL is not None:
    command += ["--attn_impl", ATTN_IMPL]

# Auto-resume: continue from the last checkpoint on Drive instead of starting over.
last_checkpoint = find_last_checkpoint(OUTPUT_DIR)
if last_checkpoint is not None:
    print("Continuing from checkpoint:", last_checkpoint)
    command += ["--resume_from_checkpoint", last_checkpoint]
else:
    print("Starting from the beginning.")

print(" ".join(command))
print("\n>>> The model is already downloaded (step 6). swift now LOADS it + preprocesses the data\n"
      ">>> - a few SILENT minutes, NOT frozen - then loss logs print every 5 steps. Liveness check\n"
      ">>> without touching this busy kernel: watch Drive Thesis/pan924_runs/<model>/ in your browser.\n")
# Stream swift's output line-by-line so you can see it is making progress (not hung).
proc = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end=''); sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit('Training failed with exit code ' + str(proc.returncode))
# CUDA OOM: lower MAX_PIXELS (1003520 -> 802816 -> 602112) and re-run (it resumes).
# The default batch (1) is the safe floor; only raise it if memory is comfortable.


## 8. Pick the best checkpoint by macro-F1 (not by loss)
Loss is dominated by the common `H` class, so we score every checkpoint on the validation set and keep the one with the best per-condition macro-F1.

In [ ]:
import os
import glob
import json
import subprocess

VAL_DATA = 'vlm_report_dataset/converted/qwen/val.jsonl'
EVAL_SCRIPT = 'vlm_report_dataset/scripts/eval_report.py'
ADAPTER_SCRIPT = 'vlm_report_dataset/training/tools/swift_pred_to_eval.py'

def list_checkpoints(folder):
    pairs = []
    for path in glob.glob(os.path.join(folder, 'checkpoint-*')):
        number_text = os.path.basename(path).replace('checkpoint-', '')
        if number_text.isdigit():
            pairs.append((int(number_text), path))
    pairs.sort()
    return [path for step, path in pairs]

env = os.environ.copy()
if USE_MAX_PIXELS:
    env['MAX_PIXELS'] = str(MAX_PIXELS)

scores = {}
for checkpoint in list_checkpoints(OUTPUT_DIR):
    infer_out = os.path.join(checkpoint, 'infer_val.jsonl')
    pred_out = os.path.join(checkpoint, 'preds_val.jsonl')
    metrics_out = os.path.join(checkpoint, 'metrics_val.json')
    subprocess.run(['swift', 'infer', '--model', MODEL_NAME,
                    '--adapters', checkpoint, '--val_dataset', VAL_DATA,
                    '--max_new_tokens', '1024', '--temperature', '0',
                    '--result_path', infer_out], env=env, check=True)
    subprocess.run(['python', ADAPTER_SCRIPT, '--val', VAL_DATA,
                    '--swift-result', infer_out, '--out', pred_out], check=True)
    subprocess.run(['python', EVAL_SCRIPT, '--gold', VAL_DATA, '--pred', pred_out,
                    '--out-json', metrics_out, '--tag', MODEL_KEY + '/val'], check=True)
    macro_f1 = json.load(open(metrics_out, encoding='utf-8'))['macro_f1']
    scores[checkpoint] = macro_f1
    print(checkpoint, 'macro-F1 =', round(macro_f1, 4))

BEST_CHECKPOINT = max(scores, key=scores.get)
print('\nBest checkpoint:', BEST_CHECKPOINT, '-> macro-F1', round(scores[BEST_CHECKPOINT], 4))


## 9. Final metrics on the test set
Reports accuracy / precision / recall / F1 (per condition and overall), FDI detection, hallucination and miss rates, exact-report match, and ROUGE-L on the text.

In [ ]:
import os
import json
import subprocess
import pandas as pd

TEST_DATA = 'vlm_report_dataset/converted/qwen/test.jsonl'
EVAL_SCRIPT = 'vlm_report_dataset/scripts/eval_report.py'
ADAPTER_SCRIPT = 'vlm_report_dataset/training/tools/swift_pred_to_eval.py'

env = os.environ.copy()
if USE_MAX_PIXELS:
    env['MAX_PIXELS'] = str(MAX_PIXELS)

infer_out = os.path.join(BEST_CHECKPOINT, 'infer_test.jsonl')
pred_out = os.path.join(BEST_CHECKPOINT, 'preds_test.jsonl')
metrics_out = os.path.join(OUTPUT_DIR, 'metrics_' + MODEL_KEY + '_test.json')

subprocess.run(['swift', 'infer', '--model', MODEL_NAME,
                '--adapters', BEST_CHECKPOINT, '--val_dataset', TEST_DATA,
                '--max_new_tokens', '1024', '--temperature', '0',
                '--result_path', infer_out], env=env, check=True)
subprocess.run(['python', ADAPTER_SCRIPT, '--val', TEST_DATA,
                '--swift-result', infer_out, '--out', pred_out], check=True)
subprocess.run(['python', EVAL_SCRIPT, '--gold', TEST_DATA, '--pred', pred_out,
                '--out-json', metrics_out, '--tag', MODEL_KEY + '/test'], check=True)

metrics = json.load(open(metrics_out, encoding='utf-8'))
for name, value in metrics.items():
    if isinstance(value, (int, float)):
        print(name, '=', round(value, 4))

# per-condition table (does the model handle the rare conditions?)
pd.DataFrame(metrics['per_condition']).T.sort_values('support', ascending=False)


## 10. Compare all models
After all 5 notebooks finish, every `metrics_<key>_test.json` is on Drive under `Thesis/pan924_runs/<model>/`:
```bash
python vlm_report_dataset/training/tools/compare_models.py /content/drive/MyDrive/Thesis/pan924_runs/*/metrics_*_test.json
```